<a href="https://colab.research.google.com/github/aleks111sh/data-science-healthcare-AH/blob/main/Copy_of_DSRP_Day_7_Lab_Notebook_v2_%5BAlexis_Hernandez%5D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Day 7: Applying Hypothesis Tests (t-tests, ANOVA, Chi-Squared)**
---

### **Description**
In this notebook, you will practice applying **three statistical tests**:

- **t-tests** (one-sample, two-sample, paired)
- **ANOVA** (+ Tukey HSD)
- **Chi-squared** tests



### **Reminder: The 4 Steps of Hypothesis Testing**
1. Write the null and alternative hypotheses (H₀ and Hₐ).
2. Choose the statistic test to use.
3. Compute the test statistic & p-value.
4. Interpret the result.

<br>

### **Resources**
* [Alternative Hypothesis Testing Cheat Sheet](https://docs.google.com/document/d/13117aJ2snJ2goIHd7I85r4UpIMl_DeMk0nu5P1-NyaA/edit?usp=sharing)

<br>


In [ ]:
## import required libraries
import numpy as np
import pandas as pd
import scipy.stats as stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd

<a id='p1'></a>
## **Part 1: One-Sample T-Test Review**
---


### **Problem #1.1**

A company claims their new phone’s battery lasts **30 hours on average**.

You test a small sample of phones and record their battery life (hours).

**Write your null and alternative hypotheses and calculate the p-value.**
*   H₀: The small sample of phones last 30 hours on average.
*   Hₐ: The small sample of phones don't last 30 hours on average.


In [ ]:
battery_hours = np.array([28, 31, 29, 30, 27, 32, 30, 29, 31, 28])

result = stats.ttest_1samp(battery_hours, popmean=30)
p_val = result.pvalue

print(p_val)


0.34343639613791355


 **Is this difference statistically significant?**

The difference is not statistically significant, as there was a 0.34 probability that the test statistic was acquired, assuming the null hypothesis is true.

Since p_val > 0.05, we fail to reject the null hypothesis. There is no significance difference between the sample mean and the stated battery life.

<a id='p1'></a>
## **Part 2: Using Datasets**
---


So far we’ve been using arrays. In real projects, your data will usually be in a **dataset (dataframe)**.

Here is a small dataset about food delivery orders.  
- `cost` is **numeric** (what we measure)  
- `platform` is **categorical** (how we group)

Run the cell below to create the dataset.


In [ ]:
# Create a small dataset
food_orders = pd.DataFrame({
    "cost":     [22, 25, 19, 28, 24, 21, 20, 26, 23, 18],
    "platform": ["iOS","iOS","Android","iOS","Android","Android","Android","iOS","iOS","Android"]
})
food_orders


,cost,platform
0,22,iOS
1,25,iOS
2,19,Android
3,28,iOS
4,24,Android
5,21,Android
6,20,Android
7,26,iOS
8,23,iOS
9,18,Android


We often need to pull out the values for each group as separate arrays.

We can do this using one of two main methods:
- **boolean filtering with .loc**
- **grouping with .groupby**

We will review both, and you can choose which of these methods you prefer, but the rest of this lab notebook will use the .groupby method.


In [ ]:
# Method 1: boolean filtering with .loc
## CODE TOGETHER
ioscost = food_orders.loc[food_orders['platform'] == 'iOS', 'cost']
androidcost = food_orders.loc[food_orders['platform'] == 'Android', 'cost']

# Method 2: grouping with .groupby
## CODE TOGETHER
groups = food_orders.groupby('platform')['cost']
ios2 = groups.get_group('iOS')
android2 = groups.get_group('Android')

<a id='p2'></a>
## **Part 3: Two-Sample T-Test (Independent Groups)**
---


### **Problem #3.1**

**Question:** Do iOS and Android users spend **different** amounts per delivery order?

Use the dataset `food orders` from above.



**Write your hypotheses here**
*   H₀: iOS and Android users do not spend different amounts per delivery order.
*   Hₐ: iOS and Android users do spend different amounts per delivery order.


In [ ]:
# Run a two-sample t-test using ios_costs and android_costs (two-tailed)
## CODE TOGETHER
result = stats.ttest_ind(ios2, android2)
t = result.statistic
p_val = result.pvalue
print("p =", p_val, ',', "t =", t)

p = 0.01796748744726236 , t = 2.9664793948382666


Is this p-value significant? How can you interpret the results?

Assuming the null hypothesis to be true (iOS and Android users do not spend different amounts per delivery order), the probability that a test statistic of 2.97 or more extreme was acquired purely by chance is approximately 0.02. As 0.02 is less than 0.05, we reject the null hypothesis and do find convincing evidence that iOS and Android users spend different amounts per delivery order.

### **Problem #3.2**

A restaurant owner believes delivery orders cost **more** than dine-in orders.

The owner collects data from a few customers:

In [ ]:
restaurant_orders = pd.DataFrame({
    "cost": [22, 27, 19, 30, 18, 28, 23, 24, 21, 26, 20, 29],
    "order_type": ["Delivery", "Delivery", "Dine-in", "Delivery",
                   "Dine-in", "Dine-in", "Delivery", "Dine-in",
                   "Delivery", "Delivery", "Dine-in", "Dine-in"]
})
restaurant_orders

,cost,order_type
0,22,Delivery
1,27,Delivery
2,19,Dine-in
3,30,Delivery
4,18,Dine-in
5,28,Dine-in
6,23,Delivery
7,24,Dine-in
8,21,Delivery
9,26,Delivery


**Write your hypotheses here**
*   H₀: Delivery orders cost the same as dine-in orders.
*   Hₐ: Delivery orders cost more than dine-in orders.


In [ ]:
# Run a two-sample t-test using Delivery and Dine-in (one-tailed)
## CODE TOGETHER
groups = restaurant_orders.groupby('order_type')['cost']
delivery = groups.get_group('Delivery')
dine = groups.get_group('Dine-in')
result = stats.ttest_ind(delivery, dine, alternative="greater")
t_s = result.statistic
p_val = result.pvalue
print("p =", p_val,",", "t =", t_s)

p = 0.23004139512924865 , t = 0.7682733253465355


Is this p-value significant? How can you interpret the results?

Assuming the null hypothesis to be true (delivery orders cost the same as dine-in orders), the probability that a test statistic of 0.77 or more extreme was acquired purely by chance is approximately 0.23. As 0.23 is greater than 0.05, we fail to reject the null hypothesis and do not find convincing evidence that delivery orders cost more than dine-in orders.

### **Problem #3.3**

A budgeting app claims users spend **less** per month than non-users. The following data comes from people surveyed about their spending and app usage:

In [ ]:
app_usage = pd.DataFrame({
    "monthly_spending": [1850, 1720, 1900, 1780, 1650, 1820, 1750, 1880,
                         2100, 2250, 2050, 2180, 2300, 2200, 2150, 2080],
    "uses_app": ["Yes", "Yes", "Yes", "Yes","Yes", "Yes", "Yes", "Yes",
                 "No", "No", "No", "No","No", "No", "No", "No"]
})
app_usage

,monthly_spending,uses_app
0,1850,Yes
1,1720,Yes
2,1900,Yes
3,1780,Yes
4,1650,Yes
5,1820,Yes
6,1750,Yes
7,1880,Yes
8,2100,No
9,2250,No


**Write your hypotheses here**
*   H₀: Users spend the same per month when compared to non-users.
*   Hₐ: Users spend less per month than non-users


In [ ]:
# Run a two-sample t-test comparing spending from users and non-users
## Step 1: Group and subset data
## YOUR CODE HERE
groups = app_usage.groupby('uses_app')['monthly_spending']
yes_app = groups.get_group('Yes')
no_app = groups.get_group('No')
## Step 2: Run a t-test
## YOUR CODE HERE
result = stats.ttest_ind(yes_app, no_app, alternative='less')
t_s = result.statistic
p_val = result.pvalue
## Step 3: Evaluate Results
print("p-value:", p_val, ",", "t-statistic:", t_s)

p-value: 2.7447236772380633e-07 , t-statistic: -8.645198605728408


Is this p-value significant? How can you interpret the results?

Assuming the null hypothesis to be true (users spend the same per month when compared to non-users), the probability that a test statistic of -8.645 or more extreme was acquired purely by chance is approximately 0. As 0 is less than 0.05, we reject the null hypothesis and do find convincing evidence that users spend less per month than non-users.



### **Problem #3.4**

A psychologist wants to know whether there is a difference in average hours of sleep between remote workers and in-office workers.

The psychologist collects data from a few study participants:

In [ ]:
sleep = pd.DataFrame({
    "hours_sleep": [7.8, 6.1, 8.4, 5.9, 7.2, 6.5, 8.9, 5.4,
                    6.8, 7.5, 5.7, 8.1, 6.3, 7.9, 5.2, 8.6,
                    6.0, 7.4, 5.8, 8.3],
    "work_type": ["Remote", "In-Office", "Remote", "In-Office",
                  "Remote", "In-Office", "Remote", "In-Office",
                  "In-Office", "Remote", "In-Office", "Remote",
                  "In-Office", "Remote", "In-Office", "Remote",
                  "In-Office", "Remote", "In-Office", "Remote"]
})
sleep

,hours_sleep,work_type
0,7.8,Remote
1,6.1,In-Office
2,8.4,Remote
3,5.9,In-Office
4,7.2,Remote
5,6.5,In-Office
6,8.9,Remote
7,5.4,In-Office
8,6.8,In-Office
9,7.5,Remote


**Write your hypotheses here**
*   H₀: There is no difference in hours of sleep between in-office and remote workers.
*   Hₐ: There is a difference in hours of sleep between in-office and remote workers.


In [ ]:
# Run a two-sample t-test comparing hours of sleep between remote and in-person workers.
## Step 1: Group and subset data
## YOUR CODE HERE
groups = sleep.groupby('work_type')['hours_sleep']
office = groups.get_group('In-Office')
remote = groups.get_group('Remote')
## Step 2: Run a t-test
## YOUR CODE HERE
result = stats.ttest_ind(office, remote)
t_s = result.statistic
p_val = result.pvalue
## Step 3: Evaluate Results
print("p-value:", p_val,",","t-statistic:",t_s)

p-value: 6.27344268366226e-08 , t-statistic: -8.787807861481122


Is this p-value significant? How can you interpret the results?

Assuming the null hypothesis to be true (there is a difference in hours of sleep between in-office and remote workers), the probability that a test statistic of -8.788 or more extreme was acquired purely by chance is approximately 0. As 0 is less than 0.05, we reject the null hypothesis and do find convincing evidence that there is a difference in hours of sleep between in-office and remote workers.

<a id='p3'></a>
## **Part 4: Paired T-Test**
---


### **Problem #4.1 (DO TOGETHER — Two-Tailed)**

**Question:** Does daily screen time change after deleting a social media app?

In [ ]:
screen_time_change = pd.DataFrame({
    "participant_id": [1,2,3,4,5,6,7,8,9,10,11,12],
    "before_hours":   [5.8, 6.5, 4.9, 7.2, 6.1, 5.4, 8.0, 6.8, 5.6, 7.5, 4.7, 6.3],
    "after_hours":    [4.6, 6.7, 4.9, 6.1, 5.2, 5.4, 6.9, 7.0, 4.9, 6.3, 4.8, 5.5]
})

screen_time_change

,participant_id,before_hours,after_hours
0,1,5.8,4.6
1,2,6.5,6.7
2,3,4.9,4.9
3,4,7.2,6.1
4,5,6.1,5.2
5,6,5.4,5.4
6,7,8.0,6.9
7,8,6.8,7.0
8,9,5.6,4.9
9,10,7.5,6.3


**Write your hypotheses here**
*   H₀: Daily screen time does not change after deleting a social media app.
*   Hₐ: Daily screen time does change after deleting a social media app.

In [ ]:
# Run a two-sample paired t-test comparing screen time hours before and after deleting a social media app.
## CODE TOGETHER
predelete = screen_time_change['before_hours']
postdelete = screen_time_change['after_hours']
result = stats.ttest_rel(predelete, postdelete)
t_s = result.statistic
p_val = result.pvalue
print('p=',p_val,",","t=",t_s)

p= 0.008627558546630295 , t= 3.18862971195269


Is this p-value significant? How can you interpret the results?

Assuming the null hypothesis to be true (daily screen time does not change after deleting a social media app), the probability that a test statistic of 3.189 or more extreme was acquired purely by chance is approximately 0.009. As 0.009 is less than 0.05, we reject the null hypothesis and do find convincing evidence that daily screen time does change after deleting a social media app.

### **Problem #4.2**

**Question:** Do athlete's 100m sprint times **decrease** after 8 weeks of training?


In [ ]:
sprint_times = pd.DataFrame({
    "name": ["Usain", "Florence", "Jesse", "Allyson", "Mo", "Sonic", "Forrest", "Diana", "Flash", "Steve", "Eliud", "Joan"],
    "pre_train_time": [10.5, 12.8, 13.1, 12.6, 12.9, 11.9, 13.3, 12.7, 13.0, 12.5, 13.2, 12.9],
    "post_train_time":  [9.9, 12.5, 12.8, 12.4, 12.6, 11.9, 13.0, 12.5, 12.7, 12.3, 12.9, 13.0]
})

sprint_times

,name,pre_train_time,post_train_time
0,Usain,10.5,9.9
1,Florence,12.8,12.5
2,Jesse,13.1,12.8
3,Allyson,12.6,12.4
4,Mo,12.9,12.6
5,Sonic,11.9,11.9
6,Forrest,13.3,13.0
7,Diana,12.7,12.5
8,Flash,13.0,12.7
9,Steve,12.5,12.3


**Write your hypotheses here**   
*   H₀: 100m sprint times do not decrease after 8 weeks of training.
*   Hₐ: 100m sprint times do decrease after 8 weeks of training.

In [ ]:
# Run a two-sample paired t-test comparing sprint times before and after training.
## CODE TOGETHER
pretrain = sprint_times['pre_train_time']
posttrain = sprint_times['post_train_time']
result = stats.ttest_rel(pretrain, posttrain, alternative="greater")
t_s = result.statistic
p_val = result.pvalue
print('p=',p_val,",","t=",t_s)

p= 0.0002597261877649541 , t= 4.839447609444384


Is this p-value significant? How can you interpret the results?

Assuming the null hypothesis to be true (100m sprint times do not decrease after 8 weeks of training), the probability that a test statistic of 4.839 or more extreme was acquired purely by chance is approximately 0.0003. As 0.0003 is less than 0.05, we reject the null hypothesis and do find convincing evidence that 100m sprint times do decrease after 8 weeks of training.

### **Problem #4.3**

A nutritionist believes participants will eat more fiber per day after attending a healthy eating workshop.

They send a survey to participants after the workshop to collect data on their fiber intake.


In [ ]:
fiber_intake = pd.DataFrame({
    "name": ["Maya", "Liam", "Ava", "Noah", "Isabella", "Ethan", "Sophia", "Lucas", "Olivia", "Mason", "Emma", "James"],
    "fiber_before": [18, 22, 25, 20, 24, 19, 23, 17, 26, 17, 20, 22],
    "fiber_after":  [18, 23, 27, 22, 25, 20, 20, 19, 25, 22, 19, 28]
})

fiber_intake

,name,fiber_before,fiber_after
0,Maya,18,18
1,Liam,22,23
2,Ava,25,27
3,Noah,20,22
4,Isabella,24,25
5,Ethan,19,20
6,Sophia,23,20
7,Lucas,17,19
8,Olivia,26,25
9,Mason,17,22


**Write your hypotheses here**   
*   H₀: Participants will not eat more fiber after attending a healthy eating workshop.
*   Hₐ: Participants will eat more fiber after attending a healthy eating workshop.

In [ ]:
# Run a two-sample paired t-test comparing fiber intake before and after the workshop.
## Step 1: Extract groups
# YOUR CODE HERE
pre_health = fiber_intake['fiber_before']
post_health = fiber_intake['fiber_after']
## Step 2: Run paired t-test
# YOUR CODE HERE
result = stats.ttest_rel(pre_health, post_health, alternative='less')
t_s = result.statistic
p_val = result.pvalue
## Step 3: Evaluate results
print("p-value:", p_val,",","t-statistic",t_s)


p-value: 0.055008108172325446 , t-statistic -1.7383837384127885


Is this p-value significant? How can you interpret the results?

Assuming the null hypothesis to be true (participants will not eat more fiber after attending a healthy eating workshop), the probability that a test statistic of -1.738 or more extreme was acquired purely by chance is approximately 0.055. As 0.055 is greater than 0.05, we fail to reject the null hypothesis and do not find convincing evidence that participants will eat more fiber after attending a healthy eating workshop.

### **Problem #4.4**

A neuroscience lab wants to know whether mice’s maze completion time (seconds) differs before and after a mild chronic stress protocol.


In [ ]:
maze_times = pd.DataFrame({
    "mouse_id": ["M1", "M2", "M3", "M4", "M5", "M6", "M7", "M8", "M9", "M10", "M11", "M12"],
    "before_stress": [42, 38, 45, 50, 47, 41, 44, 39, 46, 43, 48, 40],
    "after_stress":  [49, 36, 52, 55, 45, 44, 50, 37, 53, 46, 54, 38]
})

maze_times

,mouse_id,before_stress,after_stress
0,M1,42,49
1,M2,38,36
2,M3,45,52
3,M4,50,55
4,M5,47,45
5,M6,41,44
6,M7,44,50
7,M8,39,37
8,M9,46,53
9,M10,43,46


**Write your hypotheses here**   
*   H₀: Mice's maze completion time will not differ from before and after a mild chronic stress protocol.
*   Hₐ: Mice's maze completion time will differ from before and after a mild stress chronic stress protocol.

In [ ]:
# Run a two-sample paired t-test comparing maze times before and after a stress protocol.
## CODE TOGETHER
prestress = maze_times['before_stress']
poststress = maze_times['after_stress']
result = stats.ttest_rel(prestress, poststress)
t_s = result.statistic
p_val = result.pvalue
print('p=',p_val,",","t=",t_s)

p= 0.0228488912338581 , t= -2.643527055681036


Is this p-value significant? How can you interpret the results?

Assuming the null hypothesis to be true (mice's maze completion time will differ from before and after a mild chronic stress protocol), the probability that a test statistic of -2.644 or more extreme was acquired purely by chance is approximately 0.02. As 0.02 is less than 0.05, we reject the null hypothesis and do find convincing evidence that mice's maze completion time will differ from before and after a mild stress chronic stress protocol.

<a id='p5'></a>
## **Part 5: ANOVA + Tukey HSD**
---

Use ANOVA when comparing a numeric measurement across **3 or more groups**.


### **Problem #5.1**

**Question:** Does average average monthly gas spending differ across vehicle type (Sedan, SUV, Truck)?

In [ ]:
gas_spending = pd.DataFrame({
    "monthly_gas_spending": [
        175, 110, 182, 125, 170, 169, 118, 180,
        176, 132, 173, 165, 120, 178, 171, 115,
        179, 122, 185, 128, 168, 174, 172, 177
    ],
    "vehicle_type": [
        "SUV", "Sedan", "Truck", "Sedan",
        "SUV", "Truck", "Sedan", "SUV",
        "Truck", "Sedan", "Truck", "SUV",
        "Sedan", "SUV", "Truck", "Sedan",
        "Truck", "Sedan", "Truck", "Sedan",
        "SUV", "SUV", "SUV", "Truck"
    ]
})

gas_spending

,monthly_gas_spending,vehicle_type
0,175,SUV
1,110,Sedan
2,182,Truck
3,125,Sedan
4,170,SUV
5,169,Truck
6,118,Sedan
7,180,SUV
8,176,Truck
9,132,Sedan


**Write your hypotheses here**
*   H₀: Average monthly gas spending does not differ across vehicle types.
*   Hₐ: Average monthly gas spending does differ across vehicle types.


In [ ]:
# Run an ANOVA to compare monthly gas spending across vehicle types.
## CODE TOGETHER
groups = gas_spending.groupby('vehicle_type')['monthly_gas_spending']
suv = groups.get_group('SUV')
sedan = groups.get_group('Sedan')
truck = groups.get_group('Truck')
result = stats.f_oneway(suv, sedan, truck)
f_s = result.statistic
p_val = result.pvalue
print("p=",p_val,",","f=",f_s)
tukey = pairwise_tukeyhsd(endog=gas_spending['monthly_gas_spending'],groups=gas_spending['vehicle_type'])
print(tukey)
tukey.pvalues

p= 9.561980648479254e-15 , f= 216.68267929634536
 Multiple Comparison of Means - Tukey HSD, FWER=0.05  
group1 group2 meandiff p-adj   lower    upper   reject
------------------------------------------------------
   SUV  Sedan    -51.5    0.0 -58.9762 -44.0238   True
   SUV  Truck     3.75 0.4302  -3.7262  11.2262  False
 Sedan  Truck    55.25    0.0  47.7738  62.7262   True
------------------------------------------------------


array([1.82520665e-13, 4.30194600e-01, 4.48530102e-14])

### **Problem #5.2**

An ANOVA tells us **whether any difference exists**.  
Tukey HSD tells us **which groups differ**.

Based on the Tukey output above:
- Which pairs of vehicle types have significantly different monthly gas spending?
- Write a sentence interpreting these results.


**Write your answers here**
*   Significant pairs: The pairs of vehicle types that have significantly different monthly gas spending are SUVs-Sedans and Sedans-Trucks.
*   Results interpretation: Assuming the null hypothesis is true (average monthly gas spending does not differ across vehicle types), the probability that a f-statistic of 216.7 or more extreme was acquired purely by chance is approximately 0. As 0 is less than 0.05, we reject the null hypothesis and do find convincing evidence that average monthly gas spending does differ across vehicle types. SUVS and Sedans have significantly different monthly gas spending, and Sedans and Trucks have significantly different monthly gas spending.


**Hint:**  
Look for `reject = True` in the Tukey table. Those are the significantly different pairs.


### **Problem #5.3**

**Question:** Do average tip amounts differ across days of the week?


In [ ]:
tips = pd.DataFrame({
    "tip_amount": [
        18, 21, 20, 19, 22,
        17, 23, 19, 20, 18,
        21, 24, 22, 20, 19,
        18, 25, 21, 23, 20,
        26, 24, 27, 23, 25
    ],
    "day": [
        "Monday", "Tuesday", "Wednesday", "Thursday", "Monday",
        "Tuesday", "Friday", "Wednesday", "Thursday", "Monday",
        "Tuesday", "Friday", "Wednesday", "Thursday", "Monday",
        "Tuesday", "Friday", "Wednesday", "Thursday", "Monday",
        "Friday", "Friday", "Friday", "Friday", "Friday"
    ]
})
print(tips)


    tip_amount        day
0           18     Monday
1           21    Tuesday
2           20  Wednesday
3           19   Thursday
4           22     Monday
5           17    Tuesday
6           23     Friday
7           19  Wednesday
8           20   Thursday
9           18     Monday
10          21    Tuesday
11          24     Friday
12          22  Wednesday
13          20   Thursday
14          19     Monday
15          18    Tuesday
16          25     Friday
17          21  Wednesday
18          23   Thursday
19          20     Monday
20          26     Friday
21          24     Friday
22          27     Friday
23          23     Friday
24          25     Friday


**Write your hypotheses here**
*   H₀: Average tip amounts do not differ across days of the week.
*   Hₐ: Average tip amounts do differ across days of the week.


In [ ]:
# Run an ANOVA to compare tips across days of the week.
## Create groups
## YOUR CODE HERE
groups = tips.groupby('day')['tip_amount']
mon = groups.get_group('Monday')
tue = groups.get_group('Tuesday')
wed = groups.get_group('Wednesday')
thu = groups.get_group('Thursday')
fri = groups.get_group('Friday')
## Run ANOVA
result = stats.f_oneway(mon, tue, wed, thu, fri)
p_val = result.pvalue
f_s = result.statistic
print("ANOVA p-value:",p_val,",","ANOVA f-statistic:",f_s)

## IF p_val < 0.05, do a tukey to see which groups are significantly different from each other
tukey = pairwise_tukeyhsd(endog = tips['tip_amount'],groups = tips['day'])

print(tukey)

## If p-values are very small, they might be displayed as 0.0.
## We can print p-values directly to get the true values.
print("pairwise p-values:",tukey.pvalues)

ANOVA p-value: 3.178839889003223e-05 , ANOVA f-statistic: 12.366136034732275
   Multiple Comparison of Means - Tukey HSD, FWER=0.05   
 group1    group2  meandiff p-adj   lower   upper  reject
---------------------------------------------------------
  Friday    Monday   -5.225 0.0001 -7.9711 -2.4789   True
  Friday  Thursday   -4.125 0.0037 -7.0748 -1.1752   True
  Friday   Tuesday   -5.375 0.0002 -8.3248 -2.4252   True
  Friday Wednesday   -4.125 0.0037 -7.0748 -1.1752   True
  Monday  Thursday      1.1 0.8438 -2.1313  4.3313  False
  Monday   Tuesday    -0.15 0.9999 -3.3813  3.0813  False
  Monday Wednesday      1.1 0.8438 -2.1313  4.3313  False
Thursday   Tuesday    -1.25 0.8054 -4.6561  2.1561  False
Thursday Wednesday      0.0    1.0 -3.4061  3.4061  False
 Tuesday Wednesday     1.25 0.8054 -2.1561  4.6561  False
---------------------------------------------------------
pairwise p-values: [1.25808780e-04 3.71967741e-03 2.14254044e-04 3.71967741e-03
 8.43810182e-01 9.99908206e-01 

Which groups have significant p-values? How can you interpret the results?

The groups that have significant p-values are Friday-Monday, Friday-Thursday, Friday-Tuesday, and Friday-Wednesday. Assuming the null hypothesis to be true (average tip amounts do not differ across days of the week), the probability that a f-statistic of 12.4 or more extreme was acquired purely by chance is approximately 0. As 0 is less than 0.05, we reject the null hypothesis and do find convincing evidence that average tip amounts do differ across days of the week. Fridays have different average tip amounts when compared to all other days of the week.

<a id='p6'></a>
## **Part 6: Chi-Squared Tests**
---

Use a chi-squared test when both the grouping variable and measured varible are categorical.


### **Problem #6.1**

**Question:** Do different age groups tend to have different most used social media apps?


In [ ]:
social_media = pd.DataFrame({
    "age_group": [
        "Teen","40s","20s","30s","50s+","Teen","20s","40s",
        "30s","50s+","Teen","20s","30s","40s","Teen","50s+",
        "20s","30s","40s","Teen","50s+","20s","30s","40s",
        "Teen","50s+","20s","30s","40s","50s+"
    ],
    "most_used_app": [
        "TikTok","Facebook","Instagram","Instagram","Facebook","TikTok","TikTok","Facebook",
        "Facebook","Facebook","Instagram","Instagram","Instagram","Instagram","TikTok","Instagram",
        "Instagram","Facebook","Instagram","TikTok","Facebook","Instagram","Instagram","Facebook",
        "Instagram","Facebook","TikTok","Instagram","Facebook","Facebook"
    ]
})

print(social_media)

   age_group most_used_app
0       Teen        TikTok
1        40s      Facebook
2        20s     Instagram
3        30s     Instagram
4       50s+      Facebook
5       Teen        TikTok
6        20s        TikTok
7        40s      Facebook
8        30s      Facebook
9       50s+      Facebook
10      Teen     Instagram
11       20s     Instagram
12       30s     Instagram
13       40s     Instagram
14      Teen        TikTok
15      50s+     Instagram
16       20s     Instagram
17       30s      Facebook
18       40s     Instagram
19      Teen        TikTok
20      50s+      Facebook
21       20s     Instagram
22       30s     Instagram
23       40s      Facebook
24      Teen     Instagram
25      50s+      Facebook
26       20s        TikTok
27       30s     Instagram
28       40s      Facebook
29      50s+      Facebook


**Write your hypotheses here**
*   H₀: Different age groups do not tend to have different most used social media apps.
*   Hₐ: Different age groups do tend to have different most used social media apps.


In [ ]:
# Run a chi-squared test to investigate the relationship between age group and most used social media app.
## CODE TOGETHER
con_table = pd.crosstab(social_media['age_group'],social_media['most_used_app'])
result = stats.chi2_contingency(con_table)
chi2_stat = result.statistic
p_val = result.pvalue
expected_table = result.expected_freq
print("p=",p_val,",","chi2=",chi2_stat)
expected_table

p= 0.003507789597221319 , chi2= 22.890442890442884


array([[2.2, 2.6, 1.2],
       [2.2, 2.6, 1.2],
       [2.2, 2.6, 1.2],
       [2.2, 2.6, 1.2],
       [2.2, 2.6, 1.2]])

How can you interpret these results?

Assuming that the null hypothesis is true (different age groups do not tend to have different most used social media apps), the probability that a chi-squared statistic of 22.89 or greater was acquired purely by chance is 0.0035. As 0.0035 is less than 0.05, we reject the null hypothesis and do find convincing evidence that different age groups do tend to have different most used social media apps.

### **Problem #6.2**

**Question:** Is diet type associated with cholesterol category?


In [ ]:
health_data = pd.DataFrame({
    "diet": [
        "Standard","Plant-Based","High-Protein","Standard","Plant-Based","Standard",
        "High-Protein","Plant-Based","Standard","High-Protein","Plant-Based","Standard",
        "High-Protein","Plant-Based","Standard","High-Protein","Plant-Based","Standard",
        "Plant-Based","High-Protein","Standard","Plant-Based","High-Protein","Standard",
        "Plant-Based","High-Protein","Standard","Plant-Based","High-Protein","Standard"
    ],
    "cholesterol_level": [
        "Elevated","Normal","Elevated","Normal","Normal","Elevated",
        "Elevated","Normal","Elevated","Elevated","Normal","Normal",
        "Elevated","Normal","Elevated","Normal","Normal","Elevated",
        "Normal","Elevated","Elevated","Normal","Elevated","Normal",
        "Normal","Elevated","Elevated","Normal","Elevated","Elevated"
    ]
})

print(health_data)

            diet cholesterol_level
0       Standard          Elevated
1    Plant-Based            Normal
2   High-Protein          Elevated
3       Standard            Normal
4    Plant-Based            Normal
5       Standard          Elevated
6   High-Protein          Elevated
7    Plant-Based            Normal
8       Standard          Elevated
9   High-Protein          Elevated
10   Plant-Based            Normal
11      Standard            Normal
12  High-Protein          Elevated
13   Plant-Based            Normal
14      Standard          Elevated
15  High-Protein            Normal
16   Plant-Based            Normal
17      Standard          Elevated
18   Plant-Based            Normal
19  High-Protein          Elevated
20      Standard          Elevated
21   Plant-Based            Normal
22  High-Protein          Elevated
23      Standard            Normal
24   Plant-Based            Normal
25  High-Protein          Elevated
26      Standard          Elevated
27   Plant-Based    

**Write your hypotheses here**
*   H₀: Diet type is not associated with cholesterol category.
*   Hₐ: Diet type is associated with cholesterol category.


In [ ]:
# Run a chi-squared test to investigate the relationship between diet and cholesterol levels.
## Create a contingency table
con_table = pd.crosstab(health_data['diet'],health_data['cholesterol_level'])
print(con_table)
## Run a chi-squared test
result = stats.chi2_contingency(con_table)
chi2 = result.statistic
## Evaluate results
# (optional) look at the table of expected values
expected_table = result.expected_freq
print(expected_table)
# Is the p-value significant?
p_val = result.pvalue
print("p-value:",p_val,",","chi2-statistic:",chi2)


cholesterol_level  Elevated  Normal
diet                               
High-Protein              8       1
Plant-Based               0      10
Standard                  8       3
[[4.8        4.2       ]
 [5.33333333 4.66666667]
 [5.86666667 5.13333333]]
p-value: 0.00014610736480008806 , chi2-statistic: 17.662337662337663


How can you interpret these results?

Assuming the null hypothesis to be true (diet type is not associated with cholesterol category), the probability that a chi-squared statistic of 17.66 was acquired purely by chance is approximately 0.0001. As 0.0001 is less than 0.05, we reject the null hypothesis and we do find convincing evidence that diet type is associated with cholesterol category.

### **Problem #6.3**

 A researcher wants to know whether responses to a statement (strongly agree, agree, no opinion, disagree, strongly disagree) are dependent on the gender of the interviewer.


In [ ]:
survey_results = pd.DataFrame({
    "interviewer_gender": [
        "Woman","Woman","Woman","Woman","Man","Woman","Man","Man","Man","Man",
        "Woman","Man","Woman","Man","Woman","Woman","Man","Woman","Man","Woman",
        "Man","Woman","Man","Woman","Man","Woman","Woman","Man","Woman","Man",
        "Woman","Man","Woman","Man","Woman","Man","Woman","Man","Woman","Man"
    ],
    "response": [
        "Agree","Agree","Strongly agree","No opinion","Disagree","Agree","No opinion","Strongly agree","Disagree","Agree",
        "Strongly agree","No opinion","Agree","Disagree","No opinion","Strongly disagree","Agree","Agree","Disagree","No opinion",
        "Strongly agree","Disagree","Agree","Strongly agree","No opinion","Agree","Disagree","Strongly disagree","Agree","No opinion",
        "Strongly agree","Agree","No opinion","Disagree","Agree","Strongly agree","No opinion","Disagree","Strongly disagree","Agree"
    ]
})

print(survey_results)

   interviewer_gender           response
0               Woman              Agree
1               Woman              Agree
2               Woman     Strongly agree
3               Woman         No opinion
4                 Man           Disagree
5               Woman              Agree
6                 Man         No opinion
7                 Man     Strongly agree
8                 Man           Disagree
9                 Man              Agree
10              Woman     Strongly agree
11                Man         No opinion
12              Woman              Agree
13                Man           Disagree
14              Woman         No opinion
15              Woman  Strongly disagree
16                Man              Agree
17              Woman              Agree
18                Man           Disagree
19              Woman         No opinion
20                Man     Strongly agree
21              Woman           Disagree
22                Man              Agree
23              

**Write your hypotheses here**
*   H₀: Responses to a statement are not dependent on gender.
*   Hₐ: Responses to a statement are dependent on gender.


In [ ]:
# Run a chi-squared test to investigate the relationship between responses and interviewer gender.

## Create a contingency table
con_table = pd.crosstab(survey_results['interviewer_gender'],survey_results['response'])
print(con_table)
## Run a chi-squared test
result = stats.chi2_contingency(con_table)
chi2 = result.statistic
## Evaluate results
# (optional) look at the table of expected values
expected_table = result.expected_freq
print(expected_table)

# Is the p-value significant?
p_val = result.pvalue
print("p-value:",p_val,",","chi2-statistic:",chi2)


response            Agree  Disagree  No opinion  Strongly agree  \
interviewer_gender                                                
Man                     5         6           4               3   
Woman                   8         2           5               4   

response            Strongly disagree  
interviewer_gender                     
Man                                 1  
Woman                               2  
[[6.175 3.8   4.275 3.325 1.425]
 [6.825 4.2   4.725 3.675 1.575]]
p-value: 0.5269396104666808 , chi2-statistic: 3.187578225172211


How can you interpret these results?

Assuming the null hypothesis to be true (responses to a statement are not dependent on gender), the probability that a chi-squared statistic of 3.19 or greater was acquired purely by chance in 0.53. As 0.53 is greater than 0.05, we fail to reject the null hypothesis and we do not find convincing evidence that responses to a statement are dependent on gender.

<a id='p7'></a>
## **Part 7: Independent Practice Choosing the Correct Test**
---

For each problem:
1. Write the null and alternative hypotheses (H₀ and Hₐ).
2. Choose the statistic test to use.
3. Compute the test statistic & p-value.
4. Interpret the result.

### **Problem #7.1**

A school counselor is curious whether students’ academic performance differs depending on how they spend most of their free time. A small sample of students report their primary hobby (gaming, sports, or arts) and their current GPA.

**Question:**

Do GPAs differ across students whose primary hobby is gaming, sports, or arts?

*Run the code cell below to generate the dataset for this problem.*

In [ ]:
student_data = pd.DataFrame({
    "gpa": [
        3.4, 2.9, 3.7, 3.1, 2.8,
        3.6, 3.2, 3.0, 3.5, 2.7,
        3.8, 3.3, 2.9, 3.6, 3.1,
        3.2, 2.8, 3.4, 3.7, 3.0,
        3.5, 3.1, 2.9, 3.6
    ],
    "primary_hobby": [
        "Gaming","Sports","Arts","Gaming","Sports",
        "Arts","Sports","Gaming","Arts","Gaming",
        "Arts","Sports","Gaming","Arts","Sports",
        "Sports","Gaming","Arts","Arts","Gaming",
        "Sports","Gaming","Sports","Arts"
    ]
})

print(student_data)

    gpa primary_hobby
0   3.4        Gaming
1   2.9        Sports
2   3.7          Arts
3   3.1        Gaming
4   2.8        Sports
5   3.6          Arts
6   3.2        Sports
7   3.0        Gaming
8   3.5          Arts
9   2.7        Gaming
10  3.8          Arts
11  3.3        Sports
12  2.9        Gaming
13  3.6          Arts
14  3.1        Sports
15  3.2        Sports
16  2.8        Gaming
17  3.4          Arts
18  3.7          Arts
19  3.0        Gaming
20  3.5        Sports
21  3.1        Gaming
22  2.9        Sports
23  3.6          Arts


**Step 1: Write your null and alternative hypotheses based on the question.**


*   H₀: GPAs do not differ across students whose primary hobbies are gaming, sports, or art.
*   Hₐ: GPAs do differ across students whose primary hobbies are gaming, sports, or art.

**Step 2: Choose which hypothesis test to use.**

| Hypothetist Test      | Number of Groups | Measured Variable Type | Function |
| --------------------- | ---------------- | ----------------- |------|
| T-test, one sample    | 1                | Numeric           | stats.ttest_1samp() |
| T-test, paired | 1 at 2 time points/conditions | Numeric | stats.ttest_rel() |
| T-test, independent | 2 | Numeric | stats.ttest_ind() |
| ANOVA                 | 3+               | Numeric           | stats.f_oneway() & pairwise_tukeyhsd()|
| Chi-squared | 2+ | Categorical | stats.chi2_contingency()|

Remember that for t-tests, if you have a **directional** (less than or greater than) alternative hypothesis, set *alternative* to "greater" or "less".

**Step 3: Compute the test statistic and p-value.**

In [ ]:
## First, group the data.
groups = student_data.groupby('primary_hobby')['gpa']
game = groups.get_group('Gaming')
sport = groups.get_group('Sports')
art = groups.get_group('Arts')
## Then, run the statistical test.
result = stats.f_oneway(game, sport, art)
f_s = result.statistic
p_val = result.pvalue
## Finally, print the p-value. Make sure to also run a tukey if your ANOVA results are significant.
print("p=",p_val,",","f=",f_s)
tukey = pairwise_tukeyhsd(endog=student_data['gpa'],groups=student_data['primary_hobby'])
print(tukey)
print("P-Values from Tukey HSD:",tukey.pvalues)

p= 7.403426130398169e-06 , f= 21.84556574923547
Multiple Comparison of Means - Tukey HSD, FWER=0.05 
group1 group2 meandiff p-adj   lower   upper  reject
----------------------------------------------------
  Arts Gaming  -0.6125    0.0 -0.8612 -0.3638   True
  Arts Sports     -0.5 0.0001 -0.7487 -0.2513   True
Gaming Sports   0.1125 0.5007 -0.1362  0.3612  False
----------------------------------------------------
P-Values from Tukey HSD: [1.06771659e-05 1.45376373e-04 5.00722006e-01]


**Step 4: Interpret the result.**

The p-value is **[less than/greater than]** 0.05, so we **[can/cannot]** reject the null hypothesis.

One sentence interpreting this result:

*   Assuming the null hypothesis to be true (GPAs do not differ across students whose primary hobbies are gaming, sports, or art), the probability that a test statistic of 21.85 or greater was acquired purely by chance is approximately 0. As 0 is less than 0.05, we reject the null hypothesis and do find convincing evidence that GPAs do differ across students whose primary hobbies are gaming, sports, or arts. The pairs that demonstrate a significant difference in GPAs are Arts-Gaming and Arts-Sports.


### **Problem #7.2**

A tech company encourages its programmers to turn on blue-light filters on their devices for one week to see whether it affects their sleep. The researchers record each programmer’s average nightly sleep duration (in hours) before and after the change.

**Question**

Does average sleep duration increase after programmers turn on blue-light filters for one week?

*Run the code cell below to generate the dataset for this problem.*

In [ ]:
blue_light_data = pd.DataFrame({
    "name": [
        "Alex","Jordan","Taylor","Morgan","Riley",
        "Casey","Jamie","Avery","Parker","Quinn",
        "Rowan","Skyler","Drew","Reese","Blake"
    ],
    "before_hours": [
        5.4, 6.8, 7.2, 5.9, 6.1,
        7.5, 6.3, 5.7, 8.1, 6.0,
        5.2, 7.8, 6.4, 6.9, 5.8
    ],
    "after_hours": [
        5.9, 6.5, 7.0, 6.4, 5.8,
        7.2, 6.8, 5.5, 7.9, 6.3,
        5.6, 8.1, 6.0, 6.6, 6.1
    ]
})

print(blue_light_data)

      name  before_hours  after_hours
0     Alex           5.4          5.9
1   Jordan           6.8          6.5
2   Taylor           7.2          7.0
3   Morgan           5.9          6.4
4    Riley           6.1          5.8
5    Casey           7.5          7.2
6    Jamie           6.3          6.8
7    Avery           5.7          5.5
8   Parker           8.1          7.9
9    Quinn           6.0          6.3
10   Rowan           5.2          5.6
11  Skyler           7.8          8.1
12    Drew           6.4          6.0
13   Reese           6.9          6.6
14   Blake           5.8          6.1


**Step 1: Write your null and alternative hypotheses based on the question.**


*   H₀: Average sleep duration does not increase after programmers turn on blue-light filters for one week.
*   Hₐ: Average sleep duration does increase after programmers turn on blue-light filters for one week.

**Step 2: Choose which hypothesis test to use.**

| Hypothetist Test      | Number of Groups | Measured Variable Type | Function |
| --------------------- | ---------------- | ----------------- |------|
| T-test, one sample    | 1                | Numeric           | stats.ttest_1samp() |
| T-test, paired | 1 at 2 time points/conditions | Numeric | stats.ttest_rel() |
| T-test, independent | 2 | Numeric | stats.ttest_ind() |
| ANOVA                 | 3+               | Numeric           | stats.f_oneway() & pairwise_tukeyhsd()|
| Chi-squared | 2+ | Categorical | stats.chi2_contingency()|

Remember that for t-tests, if you have a **directional** (less than or greater than) alternative hypothesis, set *alternative* to "greater" or "less".

**Step 3: Compute the test statistic and p-value.**

In [ ]:
## First, group the data.
preblue = blue_light_data['before_hours']
postblue = blue_light_data['after_hours']
## Then, run the statistical test.
result = stats.ttest_rel(preblue, postblue, alternative = "less")
t_s = result.statistic
p_val = result.pvalue
## Finally, print the p-value. Make sure to also run a tukey if your ANOVA results are significant.
print("p=",p_val,",","t=",t_s)

p= 0.33597223261552467 , t= -0.43253023633638715


**Step 4: Interpret the result.**

The p-value is **[less than/greater than]** 0.05, so we **[can/cannot]** reject the null hypothesis.

One sentence interpreting this result:

*   Assuming the null hypothesis to be true (average sleep duration does not increase after programmers turn on blue-light filters for one week), the probability that a test statistic of -0.433 was acquired purely by chance is approximately 0.34. As 0.34 is greater than 0.05, we fail to reject the null hypothesis and do not find convincing evidence that average sleep duration increases after programmers turn on blue-light filters for one week.

### **Problem #7.3**

A consumer behavior researcher surveys shoppers to investigate whether the way people pay influences how often they make impulse purchases. Each participant reports their primary payment method and how frequently they make impulse buys.

**Question**

Is payment method associated with impulse spending frequency?

*Run the code cell below to generate the dataset for this problem.*

In [ ]:
survey_results = pd.DataFrame({
    "payment_method": [
        "Cash","Debit","Credit","Debit","Cash","Credit","Cash","Credit","Debit",
        "Credit","Cash","Debit","Cash","Credit","Debit","Cash","Credit","Debit",
        "Cash","Credit","Debit","Credit","Cash","Debit","Credit","Cash","Debit"
    ],
    "impulse_freq": [
        "Rarely","Sometimes","Often",
        "Sometimes","Rarely","Often",
        "Sometimes","Often","Rarely",
        "Sometimes","Rarely","Sometimes",
        "Rarely","Often","Sometimes",
        "Sometimes","Rarely","Often",
        "Often","Sometimes","Rarely",
        "Often","Sometimes","Rarely",
        "Sometimes","Often","Sometimes"
    ]
})

print(survey_results)

   payment_method impulse_freq
0            Cash       Rarely
1           Debit    Sometimes
2          Credit        Often
3           Debit    Sometimes
4            Cash       Rarely
5          Credit        Often
6            Cash    Sometimes
7          Credit        Often
8           Debit       Rarely
9          Credit    Sometimes
10           Cash       Rarely
11          Debit    Sometimes
12           Cash       Rarely
13         Credit        Often
14          Debit    Sometimes
15           Cash    Sometimes
16         Credit       Rarely
17          Debit        Often
18           Cash        Often
19         Credit    Sometimes
20          Debit       Rarely
21         Credit        Often
22           Cash    Sometimes
23          Debit       Rarely
24         Credit    Sometimes
25           Cash        Often
26          Debit    Sometimes


**Step 1: Write your null and alternative hypotheses based on the question.**


*   H₀: Payment method is not associated with impulse spending frequency.
*   Hₐ: Payment method is associated with impulse spending frequency.

**Step 2: Choose which hypothesis test to use.**

| Hypothetist Test      | Number of Groups | Measured Variable Type | Function |
| --------------------- | ---------------- | ----------------- |------|
| T-test, one sample    | 1                | Numeric           | stats.ttest_1samp() |
| T-test, paired | 1 at 2 time points/conditions | Numeric | stats.ttest_rel() |
| T-test, independent | 2 | Numeric | stats.ttest_ind() |
| ANOVA                 | 3+               | Numeric           | stats.f_oneway() & pairwise_tukeyhsd()|
| Chi-squared | 2+ | Categorical | stats.chi2_contingency()|

Remember that for t-tests, if you have a **directional** (less than or greater than) alternative hypothesis, set *alternative* to "greater" or "less".

**Step 3: Compute the test statistic and p-value.**

In [ ]:
## First, group the data.
con_table = pd.crosstab(survey_results['payment_method'],survey_results['impulse_freq'])
print(con_table)
## Then, run the statistical test.
result = stats.chi2_contingency(con_table)
chi2 = result.statistic
p_val = result.pvalue
## Finally, print the p-value. Make sure to also run a tukey if your ANOVA results are significant.
expected_table = result.expected_freq
print(expected_table)
print("p=",p_val,",","chi2=",chi2)

impulse_freq    Often  Rarely  Sometimes
payment_method                          
Cash                2       4          3
Credit              5       1          3
Debit               1       3          5
[[2.66666667 2.66666667 3.66666667]
 [2.66666667 2.66666667 3.66666667]
 [2.66666667 2.66666667 3.66666667]]
p= 0.22046252338694772 , chi2= 5.727272727272728


**Step 4: Interpret the result.**

The p-value is **[less than/greater than]** 0.05, so we **[can/cannot]** reject the null hypothesis.

One sentence interpreting this result:

* Assuming the null hypothesis to be true (payment method is not associated with impulse spending frequency), the probability that a chi-squared statistic of 5.73 or greater was acquired is approximately 0.22. As 0.22 is greater than 0.05, we fail to reject the null hypothesis and we do not find convincing evidence that payment method is associated with impulse spending frequency.

In [ ]:
np.random.seed(7)

n = 20
df = pd.DataFrame({
    "study_method": (["No Study Plan"] * n) + (["Flashcards"] * n) + (["Tutoring (Weekly)"] * n),
    "exam_score": np.concatenate([
        np.random.normal(loc=72, scale=6, size=n),   # baseline
        np.random.normal(loc=75, scale=6, size=n),   # small improvement (may or may not be sig)
        np.random.normal(loc=83, scale=6, size=n)    # bigger improvement (likely sig)
    ])
})

# Optional: keep scores in realistic bounds
df["exam_score"] = df["exam_score"].clip(0, 100)

tukey = pairwise_tukeyhsd(endog=df["exam_score"], groups=df["study_method"], alpha=0.05)

print(tukey)
print(tukey.pvalues)
print([1.54799202e-03,1.70307548e-01,6.17759567e-07])

         Multiple Comparison of Means - Tukey HSD, FWER=0.05         
    group1          group2      meandiff p-adj  lower   upper  reject
---------------------------------------------------------------------
   Flashcards     No Study Plan  -3.0464 0.3048 -7.977  1.8843  False
   Flashcards Tutoring (Weekly)   9.0482 0.0001 4.1175 13.9789   True
No Study Plan Tutoring (Weekly)  12.0946    0.0 7.1639 17.0252   True
---------------------------------------------------------------------
[3.04799202e-01 1.33007548e-04 6.17759567e-07]
[0.00154799202, 0.170307548, 6.17759567e-07]


---

# End of Notebook

© 2026 Frontier Technology Institute, all rights reserved